<a id="heat-equation"></a>

$$
\text{一维热传导方程 (Heat Equation): 偏微分方程 (Partial Differential Equation, PDE)} \\ \quad \\
\text{1. 傅里叶定律 (Fourier's Law): 热量从高温流向低温, 热流密度 (单位时间单位面积的热量) 和温度梯度成正比} \\
q(x, t) = -k \frac{\partial T}{\partial x} \quad (\text{k 为热导率, 负号: 温度沿 x 升高时热量向 -x 方向流}) \\ \quad \\
\text{2. 能量守恒: 取一小段 } [x, x + \Delta x] \text{, 截面积 A, 密度 } \rho \text{, 比热容 c} \\
\text{这一小段内能的变化率 = 左边流入 - 右边流出} \\
\rho c A \Delta x \frac{\partial T}{\partial t} = A\left[q(x) - q(x + \Delta x)\right] \\
\text{两边除以 } A \Delta x \text{, 令 } \Delta x \to 0 \text{ (导数的定义):} \\
\rho c \frac{\partial T}{\partial t} = -\lim_{\Delta x \to 0}\frac{q(x + \Delta x) - q(x)}{\Delta x} = -\frac{\partial q}{\partial x} = k \frac{\partial^2 T}{\partial x^2} \\
\frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial x^2}, \quad \alpha = \frac{k}{\rho c} \ (\text{热扩散率 thermal diffusivity}) \\ \quad \\
\text{直观: } \frac{\partial^2 T}{\partial x^2} > 0 \text{ (下凸, 比两边的平均低) 的点升温, 反之降温, 温度分布越来越平滑} \\
\text{同一个方程也描述扩散 (Diffusion), 只要把 "热量" 换成 "浓度", 傅里叶定律换成菲克定律 (Fick's Law)}
$$

数学基础: [导数的定义](../math_basics/calculus.ipynb#derivative) · [偏导数](../math_basics/calculus.ipynb#partial)

<a id="heat-fourier-series"></a>

$$
\text{解析解: 分离变量法 (Separation of Variables)} \\
\text{边界条件 } T(0, t) = T(L, t) = 0 \text{ (两端接冰水), 初始条件 } T(x, 0) = f(x) \\ \quad \\
\text{1. 猜 } T(x, t) = X(x) \, \tau(t) \text{, 代入: } X \tau' = \alpha X'' \tau \implies \frac{\tau'}{\alpha \tau} = \frac{X''}{X} \\
\text{左边只和 t 有关, 右边只和 x 有关, 两者相等 → 都等于同一个常数, 记为 } -\lambda \\ \quad \\
\text{2. 空间部分: } X'' + \lambda X = 0, \ X(0) = X(L) = 0 \\
\text{特征方程 } r^2 + \lambda = 0 \text{, 要有非零解且满足两端为 0, 必须 } \lambda > 0 \text{, } X = \sin\sqrt{\lambda}x \\
X(L) = \sin\sqrt{\lambda}L = 0 \implies \sqrt{\lambda}L = n\pi \implies \lambda_n = \left(\frac{n\pi}{L}\right)^2, \ X_n = \sin\frac{n\pi x}{L}, \ n = 1, 2, \cdots \\ \quad \\
\text{3. 时间部分: } \tau' = -\alpha\lambda_n \tau \implies \tau_n = e^{-\alpha (n\pi / L)^2 t} \quad (\text{分离变量 ODE}) \\ \quad \\
\text{4. 方程是线性的, 解可以叠加: } T(x, t) = \sum_{n=1}^{\infty} b_n \sin\frac{n\pi x}{L} \, e^{-\alpha (n\pi / L)^2 t} \\ \quad \\
\text{5. 系数由初始条件 } f(x) = \sum b_n \sin\frac{n\pi x}{L} \text{ 确定 (傅里叶正弦级数 Fourier Sine Series)} \\
\text{正交性 (Orthogonality): 积化和差 } \sin A \sin B = \frac{1}{2}[\cos(A - B) - \cos(A + B)] \\
\int_0^L \sin\frac{n\pi x}{L}\sin\frac{m\pi x}{L} dx = \begin{cases} 0 & n \neq m \\ \frac{L}{2} & n = m \end{cases} \\
\text{两边乘以 } \sin\frac{m\pi x}{L} \text{ 再积分, 只剩 n = m 的项: } b_m = \frac{2}{L}\int_0^L f(x)\sin\frac{m\pi x}{L} dx \\ \quad \\
\text{例: } f(x) = x(L - x) \text{, 分部积分两次: } b_n = \frac{4L^2}{n^3\pi^3}\left[1 - (-1)^n\right] = \begin{cases} \frac{8L^2}{n^3\pi^3} & n \text{ 奇数} \\ 0 & n \text{ 偶数} \end{cases} \\
\text{高频分量 (n 大) 衰减得快 } (\propto e^{-n^2 t}) \text{, 所以温度分布很快变成一个 } \sin\frac{\pi x}{L} \text{ 的形状}
$$

数学基础: [分离变量法 (ODE)](../math_basics/calculus.ipynb#ode-separable) · [特征方程](../math_basics/calculus.ipynb#ode-system) · [分部积分](../math_basics/calculus.ipynb#integral) · [正交 (内积为 0)](../math_basics/linear_algebra.ipynb#orthogonal)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

L_rod, alpha = 1.0, 0.01
f0 = lambda x: x * (L_rod - x)

# 正交性
print('正交性矩阵 (2/L) * int sin(n pi x / L) sin(m pi x / L) dx:')
ortho = np.array([[2 / L_rod * quad(lambda x: np.sin(n * np.pi * x / L_rod) * np.sin(m * np.pi * x / L_rod), 0, L_rod)[0]
                   for m in range(1, 5)] for n in range(1, 5)])
print(np.round(ortho, 10) + 0.0)   # + 0.0 去掉 -0.

# 系数: 数值积分 vs 公式
n_terms = 60
n_idx = np.arange(1, n_terms + 1)
b_quad = np.array([2 / L_rod * quad(lambda x: f0(x) * np.sin(n * np.pi * x / L_rod), 0, L_rod)[0] for n in n_idx])
b_formula = 4 * L_rod**2 / (n_idx**3 * np.pi**3) * (1 - (-1.0)**n_idx)
print(f'前 5 个 b_n (quad) = {np.round(b_quad[:5], 6)}')
print(f'b_n: quad == 公式? {np.allclose(b_quad, b_formula, atol=1e-12)}')

def heat_series(x, t):
    modes = np.sin(np.outer(x, n_idx) * np.pi / L_rod)
    return modes @ (b_formula * np.exp(-alpha * (n_idx * np.pi / L_rod)**2 * t))

x_check = np.linspace(0, L_rod, 101)
print(f't = 0 时级数 == f(x)? 最大误差 = {np.max(np.abs(heat_series(x_check, 0) - f0(x_check))):.2e}')
for n in [1, 3, 5]:
    print(f'第 {n} 个模态的衰减时间常数 1 / (alpha (n pi / L)^2) = {1 / (alpha * (n * np.pi / L_rod)**2):.3f}')

<a id="ftcs"></a>

$$
\text{数值解: 有限差分 (Finite Difference), FTCS 格式 (Forward Time, Central Space)} \\
\text{网格 } x_j = j\Delta x, \ t_n = n\Delta t \text{, 记 } T_j^n = T(x_j, t_n) \\
\text{时间: 前向差分 } \frac{\partial T}{\partial t} \approx \frac{T_j^{n+1} - T_j^n}{\Delta t} \ (O(\Delta t)) \text{; 空间: 二阶中心差分 } \frac{\partial^2 T}{\partial x^2} \approx \frac{T_{j+1}^n - 2T_j^n + T_{j-1}^n}{\Delta x^2} \ (O(\Delta x^2)) \\
T_j^{n+1} = T_j^n + r\left(T_{j+1}^n - 2T_j^n + T_{j-1}^n\right), \quad r = \frac{\alpha \Delta t}{\Delta x^2} \\ \quad \\
\text{稳定性 (Stability): 冯·诺依曼分析 (von Neumann Analysis)} \\
\text{任何误差都可以分解成傅里叶模态, 看单个模态 } T_j^n = G^n e^{i\kappa j\Delta x} \text{ (G 为每一步的放大因子) 是否增长} \\
G^{n+1} e^{i\kappa j\Delta x} = G^n e^{i\kappa j\Delta x}\left[1 + r\left(e^{i\kappa\Delta x} - 2 + e^{-i\kappa\Delta x}\right)\right] \\
e^{i\theta} + e^{-i\theta} = 2\cos\theta \text{ (欧拉公式)} \implies G = 1 + r(2\cos\kappa\Delta x - 2) = 1 - 4r\sin^2\frac{\kappa\Delta x}{2} \\
\text{稳定要求所有模态 } |G| \leq 1 \text{: } G \leq 1 \text{ 总成立; } G \geq -1 \iff 4r\sin^2\frac{\kappa\Delta x}{2} \leq 2 \\
\text{最坏的情况 } \sin^2 = 1 \text{ (最高频, 锯齿状的模态)} \implies r = \frac{\alpha\Delta t}{\Delta x^2} \leq \frac{1}{2} \\
\text{代价: 空间网格加密一倍, 时间步长必须缩小到 1/4}
$$

数学基础: [数值微分 (二阶导的中心差分)](../math_basics/calculus.ipynb#numerical-diff) · [欧拉法](../math_basics/calculus.ipynb#euler-rk4) · [欧拉公式](../math_basics/calculus.ipynb#taylor)

In [ ]:
def ftcs(f, nx, r, t_end):
    x = np.linspace(0, L_rod, nx + 1)
    dx = x[1] - x[0]
    dt = r * dx**2 / alpha
    n_steps = int(round(t_end / dt))
    T = f(x).astype(float)
    for _ in range(n_steps):
        T[1:-1] = T[1:-1] + r * (T[2:] - 2 * T[1:-1] + T[:-2])   # 右边先全部算好再赋值, 用的都是第 n 步的值
        T[0] = T[-1] = 0.0
    return x, T, n_steps * dt

t_end = 5.0
for r in [0.25, 0.4, 0.5]:
    x, T_num, t_real = ftcs(f0, 50, r, t_end)
    print(f'r = {r}: t = {t_real:.4f}, 和傅里叶级数解的最大误差 = {np.max(np.abs(T_num - heat_series(x, t_real))):.2e}')

# 网格收敛: dx 减半, 误差约变成 1/4 (r 固定 => dt 正比于 dx^2, 总误差 O(dx^2))
for nx in [10, 20, 40, 80]:
    x, T_num, t_real = ftcs(f0, nx, 0.25, 2.0)
    print(f'nx = {nx:>2}: 最大误差 = {np.max(np.abs(T_num - heat_series(x, t_real))):.3e}')

# 不稳定: r = 0.52, 初始加入一个极小的锯齿扰动 (最高频模态)
x, T_bad, _ = ftcs(lambda x: f0(x) + 1e-10 * (-1.0)**np.arange(len(x)), 50, 0.52, t_end)
G_worst = 1 - 4 * 0.52
print(f'r = 0.52: 最高频模态放大因子 G = {G_worst:.2f}, 结果最大值 = {np.max(np.abs(T_bad)):.3e} (发散)')

# 放大因子: 数值测一个模态每一步的放大倍数
nx, r = 50, 0.4
x = np.linspace(0, L_rod, nx + 1)
for m in [1, 10, 49]:
    mode = np.sin(m * np.pi * x / L_rod)
    step = mode.copy()
    step[1:-1] = mode[1:-1] + r * (mode[2:] - 2 * mode[1:-1] + mode[:-2])
    G_num = step[nx // 3] / mode[nx // 3]
    print(f'模态 m = {m:>2}: 数值放大因子 = {G_num:.6f}, 公式 1 - 4r sin^2(k dx / 2) = {1 - 4 * r * np.sin(m * np.pi / nx / 2)**2:.6f}')

plt.figure(figsize=(4, 3))
for t_show in [0, 1, 3, 10, 30]:
    x, T_num, t_real = ftcs(f0, 50, 0.4, t_show)
    plt.plot(x, T_num, label=f't = {t_real:.1f}')
plt.xlabel('x')
plt.ylabel('T')
plt.title('heat equation (FTCS)')
plot_nicely()
plt.show()

<a id="wave-equation"></a>

$$
\text{一维波动方程 (Wave Equation): 拉紧的弦 (string)} \\ \quad \\
\text{推导: 弦的张力 } T_s \text{ (各处相同), 线密度 } \mu \text{, 横向位移 } u(x, t) \text{, 斜率很小 (小振动)} \\
\text{取一小段 } [x, x + \Delta x] \text{, 两端张力沿切线方向, 竖直分量 } T_s\sin\theta \approx T_s\tan\theta = T_s \frac{\partial u}{\partial x} \\
\text{竖直方向合力 } = T_s\left[\frac{\partial u}{\partial x}(x + \Delta x) - \frac{\partial u}{\partial x}(x)\right] \approx T_s \frac{\partial^2 u}{\partial x^2}\Delta x \\
\text{牛顿第二定律 (质量 } \mu\Delta x \text{): } \mu\Delta x \frac{\partial^2 u}{\partial t^2} = T_s \frac{\partial^2 u}{\partial x^2}\Delta x \\
\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}, \quad c = \sqrt{\frac{T_s}{\mu}} \ (\text{波速}) \\ \quad \\
\text{达朗贝尔解 (d'Alembert's Solution): 换元 } \xi = x - ct, \ \eta = x + ct \text{, 用多元链式法则} \\
u_x = u_\xi + u_\eta, \quad u_{xx} = u_{\xi\xi} + 2u_{\xi\eta} + u_{\eta\eta} \\
u_t = -c u_\xi + c u_\eta, \quad u_{tt} = c^2\left(u_{\xi\xi} - 2u_{\xi\eta} + u_{\eta\eta}\right) \\
u_{tt} - c^2 u_{xx} = -4c^2 u_{\xi\eta} = 0 \implies u_{\xi\eta} = 0 \\
\text{对 } \eta \text{ 积分: } u_\xi = \text{只和 } \xi \text{ 有关的函数; 再对 } \xi \text{ 积分: } u = F(\xi) + G(\eta) \\
u(x, t) = F(x - ct) + G(x + ct) \quad (\text{向右和向左传播的两个波形, 形状不变}) \\
\text{初始 } u(x, 0) = f(x), \ u_t(x, 0) = 0 \text{: } F + G = f, \ -cF' + cG' = 0 \implies F = G = \frac{f}{2} \\
u(x, t) = \frac{1}{2}\left[f(x - ct) + f(x + ct)\right] \text{ (一个脉冲分成两半, 往两边走)} \\
\text{两端固定 } u(0) = u(L) = 0 \text{: 把 f 做奇延拓 (odd extension) 成周期 2L 的函数, 反射时波形上下翻转}
$$

数学基础: [多元链式法则](../math_basics/calculus.ipynb#partial) · [二阶偏导](../math_basics/calculus.ipynb#partial) · 应用: [简谐振动 (每一点都在振动)](harmonic_oscillator.ipynb#simple-harmonic)

<a id="cfl"></a>

$$
\text{数值解: 时间和空间都用二阶中心差分} \\
\frac{u_j^{n+1} - 2u_j^n + u_j^{n-1}}{\Delta t^2} = c^2 \frac{u_{j+1}^n - 2u_j^n + u_{j-1}^n}{\Delta x^2} \\
u_j^{n+1} = 2u_j^n - u_j^{n-1} + C^2\left(u_{j+1}^n - 2u_j^n + u_{j-1}^n\right), \quad C = \frac{c\Delta t}{\Delta x} \ (\text{库朗数 Courant number}) \\
\text{第一步需要 } u^{-1} \text{: 由泰勒展开 } u_j^1 = u_j^0 + \Delta t \, u_t + \frac{\Delta t^2}{2} u_{tt} = u_j^0 + \frac{C^2}{2}\left(u_{j+1}^0 - 2u_j^0 + u_{j-1}^0\right) \ (u_t = 0) \\ \quad \\
\text{稳定性: 代入 } u_j^n = G^n e^{i\kappa j\Delta x} \text{, 记 } s = \sin\frac{\kappa\Delta x}{2} \\
G = 2 - \frac{1}{G} - 4C^2 s^2 \implies G^2 - (2 - 4C^2 s^2) G + 1 = 0 \\
\text{两个根的乘积 = 1 (韦达定理), 要两个根都满足 } |G| \leq 1 \text{, 只能 } |G| = 1 \text{ (共轭复根)} \\
\iff \text{判别式} \leq 0 \iff (2 - 4C^2 s^2)^2 \leq 4 \iff 0 \leq C^2 s^2 \leq 1 \\
\text{对所有模态 } (s^2 \leq 1) \text{ 成立} \iff C = \frac{c\Delta t}{\Delta x} \leq 1 \quad (\text{CFL 条件, Courant-Friedrichs-Lewy}) \\
\text{物理意义: 一个时间步内波传播的距离 } c\Delta t \text{ 不能超过一个网格 } \Delta x \text{ (数值的依赖区域要包含真实的依赖区域)} \\
\text{特别地 } C = 1 \text{ 时差分格式和达朗贝尔解在网格点上完全一致}
$$

In [ ]:
c_wave, L_str = 1.0, 2.0
pulse = lambda x: np.exp(-((x - 0.6) / 0.08)**2)

def odd_periodic(f, x):
    # 奇延拓: 在 [-L, 0] 上 f(-x) = -f(x), 再以 2L 为周期
    y = np.mod(x, 2 * L_str)
    return np.where(y <= L_str, f(y), -f(2 * L_str - y))

def dalembert(x, t):
    return 0.5 * (odd_periodic(pulse, x - c_wave * t) + odd_periodic(pulse, x + c_wave * t))

def wave_fd(nx, C, t_end):
    x = np.linspace(0, L_str, nx + 1)
    dx = x[1] - x[0]
    dt = C * dx / c_wave
    n_steps = int(round(t_end / dt))
    u_prev = pulse(x)
    u_prev[0] = u_prev[-1] = 0
    u = u_prev.copy()
    u[1:-1] = u_prev[1:-1] + 0.5 * C**2 * (u_prev[2:] - 2 * u_prev[1:-1] + u_prev[:-2])
    for _ in range(n_steps - 1):
        u_next = np.zeros_like(u)
        u_next[1:-1] = 2 * u[1:-1] - u_prev[1:-1] + C**2 * (u[2:] - 2 * u[1:-1] + u[:-2])
        u_prev, u = u, u_next
    return x, u, n_steps * dt

for C in [1.0, 0.9, 0.5]:
    x, u, t_real = wave_fd(800, C, 1.7)
    print(f'C = {C}: t = {t_real:.4f}, 和达朗贝尔解 (奇延拓) 的最大误差 = {np.max(np.abs(u - dalembert(x, t_real))):.2e}')

x, u_bad, _ = wave_fd(800, 1.02, 1.7)
print(f'C = 1.02 (违反 CFL): 最大值 = {np.max(np.abs(u_bad)):.3e} (发散)')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for t_show in [0, 0.3, 0.8, 1.7]:
    x, u, t_real = wave_fd(800, 0.9, t_show) if t_show > 0 else (np.linspace(0, L_str, 801), pulse(np.linspace(0, L_str, 801)), 0)
    axs[0].plot(x, u, label=f't = {t_real:.2f}')
    axs[0].plot(x, dalembert(x, t_real), 'k:', linewidth=0.8)
axs[0].set_title('wave on a string (dotted: d Alembert)')
axs[0].set_xlabel('x')
s_grid = np.linspace(0, 1, 200)
for C in [0.5, 1.0, 1.1]:
    b_coef = 2 - 4 * C**2 * s_grid**2          # G^2 - b G + 1 = 0
    disc = b_coef**2 - 4
    # 判别式 <= 0: 共轭复根, |G| = 1; > 0: 两个实根, 绝对值较大的是 (|b| + sqrt(disc)) / 2 > 1
    G_abs = np.where(disc <= 0, 1.0, (np.abs(b_coef) + np.sqrt(np.maximum(disc, 0))) / 2)
    axs[1].plot(s_grid, G_abs, label=f'C = {C}')
axs[1].set_xlabel('s = sin(k dx / 2)')
axs[1].set_ylabel('max |G|')
axs[1].set_title('von Neumann amplification')
plot_nicely()
plt.show()